# RQ3: training data and exposure — final results

E7 sampling, E8 corpus eligibility and E9 episode size address distinct changes in data exposure. E8/E9 are PD-only. The exact large-table 8,192-row control is reused once. Equal optimizer updates do not imply equal compute or unique rows. Final evaluation averages five paired outer folds within table, then weights tables equally. AUC/R² gains are CPT minus original; Brier/RMSE gains reverse direction. RMSE gains here are relative percentages.

## 0. Evidence and interpretation

The selected local evidence is a snapshot. Missing evidence remains unavailable; queued VSC jobs are not completed results.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from src.visualize import style, campaign as cp, questions as rq, synthesis as sy, research_views as rv
from src.visualize import choice_effects as ce, comparisons as cmp, publication as pub, dashboards as db
from src.visualize import diagnostics as dg
from src.visualize import training_summary as ts, pilots as pv, finishing as finish, prediction_diagnostics as pred
from src.visualize.figures import FigureSaver
from src.visualize.inputs import source_description
style.apply()
sink = FigureSaver('rq3/02_results')
report = cp.NotebookReport('RQ3: training data and exposure — final results', sink=sink)
print(source_description())
report.add('0. Evidence and interpretation', source_description() + '\n' + 'E7 sampling, E8 corpus eligibility and E9 episode size address distinct changes in data exposure. E8/E9 are PD-only. The exact large-table 8,192-row control is reused once. Equal optimizer updates do not imply equal compute or unique rows. Final evaluation averages five paired outer folds within table, then weights tables equally. AUC/R² gains are CPT minus original; Brier/RMSE gains reverse direction. RMSE gains here are relative percentages.')

## 1. Coverage, protocols and unique evidence

Training completion and benchmark completion are separate. All results retain their own matched original checkpoint. The current inference-policy table governs cross-study TabICL comparisons; a pending FP32 refresh is not published evidence.

In [ ]:
QUESTION = 3
studies = {track: rq.load(QUESTION, track) for track in ('pd', 'lgd')}
catalogs = {}
for track, values in studies.items():
    report.table(track.upper() + ' unique trial inventory', rv.inventory_summary(rv.unique_training_inventory(values)))
    report.table(track.upper() + ' recipe and inference comparability', sy.comparability_table(values))
    for e, arm, run in values:
        report.table(track.upper() + ' ' + str(run.cfg.run_name) + ' ' + str(arm or '') + ' five-fold coverage', cmp.benchmark_coverage(run))
    catalogs[track] = {(domain, metric): rq.catalog(values, QUESTION, metric, domain, reference_only=QUESTION == 2)
                       for domain in ('credit', 'ood') for metric in (sy.PRIMARY[track], sy.SECONDARY[track])}
report.add('1. Coverage, protocols and unique evidence', 'Training completion and benchmark completion are separate. All results retain their own matched original checkpoint. The current inference-policy table governs cross-study TabICL comparisons; a pending FP32 refresh is not published evidence.')

## 2. Original-to-CPT effects across the research question

Held-out credit transfer (primary and secondary metric) and non-credit retention share one view. Every dot is a table, the diamond is the equal-table mean and the bar is the table interquartile range: heterogeneity, not a confidence interval. The same tables contribute to every displayed recipe within each model and column.

In [ ]:
for track in studies:
    columns = rq.effect_columns(catalogs[track], track)
    cp.show(sink, rq.plot_effects(columns, track, title='Transfer and retention by training-data choice', name='effects', min_arms=1), track=track)
    for metric in (sy.PRIMARY[track], sy.SECONDARY[track]):
        report.table(track.upper() + ' original/CPT scores and paired effects: ' + metric, sy.summarize(catalogs[track]['credit', metric]))
report.add('2. Original-to-CPT effects across the research question', 'Held-out credit transfer (primary and secondary metric) and non-credit retention share one view. Every dot is a table, the diamond is the equal-table mean and the bar is the table interquartile range: heterogeneity, not a confidence interval. The same tables contribute to every displayed recipe within each model and column.')

## 3. What changes when a specific choice changes?

Paired contrasts compare the same table and model between two planned data choices, for both credit metrics. A positive contrast can mean less damage, not positive transfer.

In [ ]:
for track in studies:
    differences = {metric: rq.choice_differences(catalogs[track]['credit', metric], QUESTION) for metric in (sy.PRIMARY[track], sy.SECONDARY[track])}
    for metric, table in differences.items():
        report.table(track.upper() + ' matched choice differences: ' + metric, table)
    columns = [('Held-out credit', table) for table in differences.values()]
    cp.show(sink, rq.plot_effects(columns, track, title='Paired differences between planned data choices', name='choices', contrast=True), track=track)
report.add('3. What changes when a specific choice changes?', 'Paired contrasts compare the same table and model between two planned data choices, for both credit metrics. A positive contrast can mean less damage, not positive transfer.')

## 4. Non-credit retention alongside transfer

The four same-task non-credit tables are a bounded retention panel, plotted with the transfer effects above. Their four training-partition results are averaged within table before equal-table summaries.

In [ ]:
for track, values in studies.items():
    for metric in (sy.PRIMARY[track], sy.SECONDARY[track]):
        report.table(track.upper() + ' non-credit absolute scores and effects: ' + metric, sy.summarize(catalogs[track]['ood', metric]))
report.add('4. Non-credit retention alongside transfer', 'The four same-task non-credit tables are a bounded retention panel, plotted with the transfer effects above. Their four training-partition results are averaged within table before equal-table summaries.')

## 5. Secondary outcomes and saved-prediction evidence

Calibration, log loss and regression errors retain their distinct meanings. Detailed per-table prediction summaries remain available without another repeated figure for each experiment. No post-hoc metric is used to choose the recipe.

In [ ]:
for track, values in studies.items():
    for e, arm, run in values:
        if e == 1:
            continue
        label = track.upper() + ' ' + str(run.cfg.run_name) + ' ' + str(arm or '')
        for metric in (('log_loss', 'ece', 'f1') if track == 'pd' else ('mae',)):
            report.table(label + ' ' + metric, cp.benchmark_effects(run, metric))
        diagnostic, support = pred.analyze(run, arm=arm)
        report.table(label + ' prediction support', support)
        report.table(label + ' prediction diagnostics', diagnostic)
report.add('5. Secondary outcomes and saved-prediction evidence', 'Calibration, log loss and regression errors retain their distinct meanings. Detailed per-table prediction summaries remain available without another repeated figure for each experiment. No post-hoc metric is used to choose the recipe.')

## 6. Cost, complete numerical results and conclusions

Transfer is model-, task- and recipe-dependent. Compare paired changes rather than raw scores across different inference settings. Missing model additions are explicit. These empirical questions were organized after results were visible; they are not preregistered directional hypotheses.

In [ ]:
for track, values in studies.items():
    primary = catalogs[track]['credit', sy.PRIMARY[track]]
    report.table(track.upper() + ' recorded cost', sy.costs(values))
    for (domain, metric), data in catalogs[track].items():
        report.table(track.upper() + ' all named-table pairs: ' + domain + ' ' + metric, data)
report.add('6. Cost, complete numerical results and conclusions', 'Transfer is model-, task- and recipe-dependent. Compare paired changes rather than raw scores across different inference settings. Missing model additions are explicit. These empirical questions were organized after results were visible; they are not preregistered directional hypotheses.')

## Complete text summary

Every displayed table, plotted value and paper caption, in section order.

In [ ]:
print(report.summary(sink))